# Deploy a Web App to a Sandbox

Upload a Node.js app, start a server, expose a port, and test the public URL.

## Prerequisites
- `az login`
- `pip install azure-sandbox azure-mgmt-sandbox`

### 0. Initialize

In [ ]:
import os, json, subprocess, time
from azure.sandbox import SandboxClient
from azure.mgmt.sandbox import SandboxGroupManagementClient

lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

account = json.loads(subprocess.run(
    ['az', 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, shell=True).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'
resource_group_location = 'westus2'
sandbox_group_name = f'{lab_name}-sg'

client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupManagementClient(subscription_id=subscription_id, resource_group=resource_group_name)
print(f'Ready: {resource_group_name} / {sandbox_group_name}')

### 1. Create resources

In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Group: {group["name"]}')

sbx = client.create_sandbox(sandbox_group_name, disk='ubuntu')
sandbox_id = sbx['id']
print(f'Sandbox: {sandbox_id}')

### 2. Upload app code

In [ ]:
app_code = """
const http = require('http');
const os = require('os');

const server = http.createServer((req, res) => {
  res.writeHead(200, {'Content-Type': 'application/json'});
  res.end(JSON.stringify({
    message: 'Hello from sandbox!',
    hostname: os.hostname(),
    uptime: process.uptime(),
    path: req.url,
  }, null, 2));
});

server.listen(8080, '0.0.0.0', () => {
  console.log('Server running on :8080');
});
"""

client.write_file(sandbox_id, sandbox_group_name, '/app/index.js', app_code.strip())
print('Uploaded /app/index.js')

### 3. Start server

In [ ]:
client.exec(sandbox_id, sandbox_group_name, 'cd /app && node index.js &')
time.sleep(2)

# Test locally inside sandbox
result = client.exec(sandbox_id, sandbox_group_name, 'curl -s http://localhost:8080')
print(f'Local test: {result["stdout"]}')

### 4. Expose port

In [ ]:
ports = client.add_port(sandbox_id, sandbox_group_name, 8080, anonymous=True)
url = None
for p in ports.get('ports', []):
    url = p.get('url')
    print(f'Public URL: {url}')

### 5. Test public URL

In [ ]:
time.sleep(3)
result = client.exec(sandbox_id, sandbox_group_name, f'curl -s {url}')
print(result['stdout'])

### 6. Clean up

In [ ]:
client.delete_sandbox(sandbox_id, sandbox_group_name)
print(f'Deleted sandbox')

mgmt.delete_group(sandbox_group_name)
print(f'Deleted group')

!az group delete --name {resource_group_name} --yes --no-wait
print(f'Deleting resource group')